In [2]:
import requests, json, time
import pandas as pd
from pathlib import Path

In [8]:
URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 10.8231,
    "longitude": 106.6297,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Ho_Chi_Minh",
    "forecast_days": 7,
}
headers = {"User-Agent": "data-pipeline-project/0.1"}

r = requests.get(URL, params=params, headers=headers, timeout=10)

print(r.status_code)
print(r.url)
print(r.headers["Content-Type"])

200
https://api.open-meteo.com/v1/forecast?latitude=10.8231&longitude=106.6297&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Asia%2FHo_Chi_Minh&forecast_days=7
application/json; charset=utf-8


In [9]:
data = r.json()
print(data.keys())
print(data["daily"])

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': ['2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-06', '2026-10-07'], 'temperature_2m_max': [32.7, 31.8, 32.0, 32.2, 32.5, 31.1, 31.1], 'temperature_2m_min': [24.3, 23.9, 24.2, 23.7, 24.5, 24.0, 23.9], 'precipitation_sum': [5.3, 7.9, 7.0, 5.7, 13.9, 4.0, 7.6]}


In [10]:
bad = dict(params, latitude=999)
r2 = requests.get(URL, params=bad, headers=headers, timeout=10)

print(r2.status_code)
print(r2.text)

400
{"reason":"Latitude must be in range of -90 to 90°. Given: 999.0.","error":true}


In [11]:
try:
    r2.raise_for_status()
except requests.HTTPError as e:
    print("Bắt được lỗi:", e)

Bắt được lỗi: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=999&longitude=106.6297&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Asia%2FHo_Chi_Minh&forecast_days=7


In [12]:
BASE = "https://jsonplaceholder.typicode.com/posts"

r3 = requests.get(BASE, params={"_page": 1, "_limit": 10}, timeout=10)
print(r3.status_code)

page1 = r3.json()
print(len(page1))
print(page1[0])

200
10
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [13]:
def fetch_all(page_size=10, max_pages=50):
    rows, page = [], 1
    while page <= max_pages:
        r = requests.get(BASE, params={"_page": page, "_limit": page_size}, timeout=10)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        rows.extend(batch)
        print(f"Trang {page}: {len(batch)} bài, tổng {len(rows)}")
        page += 1
    return rows

posts = fetch_all()
print("Tổng số bài:", len(posts))

Trang 1: 10 bài, tổng 10
Trang 2: 10 bài, tổng 20
Trang 3: 10 bài, tổng 30
Trang 4: 10 bài, tổng 40
Trang 5: 10 bài, tổng 50
Trang 6: 10 bài, tổng 60
Trang 7: 10 bài, tổng 70
Trang 8: 10 bài, tổng 80
Trang 9: 10 bài, tổng 90
Trang 10: 10 bài, tổng 100
Tổng số bài: 100


In [14]:
def get_json(url, params=None, headers=None, retries=4, timeout=10):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            if r.status_code == 429 or r.status_code >= 500:
                err = f"lỗi tạm thời, mã {r.status_code}"
            else:
                r.raise_for_status()      # 4xx khác: ném lỗi ngay, không retry
                return r.json()
        except (requests.Timeout, requests.ConnectionError) as e:
            err = f"mạng/timeout ({type(e).__name__})"

        if attempt < retries:
            wait = 2 ** attempt           # 2, 4, 8 giây
            print(f"Lần {attempt} thất bại: {err}. Chờ {wait}s rồi thử lại")
            time.sleep(wait)

    raise RuntimeError(f"Thất bại sau {retries} lần: {err}")

In [15]:
d = get_json(URL, params=params, headers=headers)
print(d["daily"]["time"])

['2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-06', '2026-10-07']


In [16]:
get_json(URL, params=dict(params, latitude=999), headers=headers)

HTTPError: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=999&longitude=106.6297&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Asia%2FHo_Chi_Minh&forecast_days=7

In [17]:
get_json("https://httpbin.org/status/503", retries=3)

Lần 1 thất bại: lỗi tạm thời, mã 503. Chờ 2s rồi thử lại
Lần 2 thất bại: lỗi tạm thời, mã 503. Chờ 4s rồi thử lại


RuntimeError: Thất bại sau 3 lần: lỗi tạm thời, mã 503

In [18]:
cities = {
    "Ha Noi":  (21.0285, 105.8542),
    "TP HCM":  (10.8231, 106.6297),
    "Da Nang": (16.0544, 108.2022),
}

frames = []
for name, (lat, lon) in cities.items():
    d = get_json(URL, params=dict(params, latitude=lat, longitude=lon), headers=headers)
    df_city = pd.DataFrame(d["daily"])
    df_city["city"] = name
    frames.append(df_city)
    print(name, "→", len(df_city), "dòng")
    time.sleep(0.5)

Ha Noi → 7 dòng
TP HCM → 7 dòng
Da Nang → 7 dòng


In [19]:
df = pd.concat(frames, ignore_index=True)
df["time"] = pd.to_datetime(df["time"])
df["ingested_at"] = pd.Timestamp.now()

print(df.shape)
print(df.dtypes)
df

(21, 6)
time                  datetime64[us]
temperature_2m_max           float64
temperature_2m_min           float64
precipitation_sum            float64
city                             str
ingested_at           datetime64[us]
dtype: object


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,city,ingested_at
0,2026-10-01,34.1,27.0,0.4,Ha Noi,2026-10-01 11:37:17.663003
1,2026-10-02,32.1,25.9,4.4,Ha Noi,2026-10-01 11:37:17.663003
2,2026-10-03,32.0,26.1,1.2,Ha Noi,2026-10-01 11:37:17.663003
3,2026-10-04,29.3,24.1,16.2,Ha Noi,2026-10-01 11:37:17.663003
4,2026-10-05,28.5,23.5,0.5,Ha Noi,2026-10-01 11:37:17.663003
5,2026-10-06,28.4,21.1,0.0,Ha Noi,2026-10-01 11:37:17.663003
6,2026-10-07,29.3,19.6,0.0,Ha Noi,2026-10-01 11:37:17.663003
7,2026-10-01,32.7,24.3,5.3,TP HCM,2026-10-01 11:37:17.663003
8,2026-10-02,31.8,23.9,7.9,TP HCM,2026-10-01 11:37:17.663003
9,2026-10-03,32.0,24.2,7.0,TP HCM,2026-10-01 11:37:17.663003


In [20]:
print(df["city"].value_counts())
print(df.isna().sum())

city
Ha Noi     7
TP HCM     7
Da Nang    7
Name: count, dtype: int64
time                  0
temperature_2m_max    0
temperature_2m_min    0
precipitation_sum     0
city                  0
ingested_at           0
dtype: int64


In [21]:
out = Path("../../data")
out.mkdir(exist_ok=True)
print(out.resolve())

D:\BI\projects\data-pipeline-project\data


In [22]:
df.to_csv(out / "weather.csv", index=False)
df.to_json(out / "weather.json", orient="records", date_format="iso", indent=2)
df.to_parquet(out / "weather.parquet", index=False)

for f in ["weather.csv", "weather.json", "weather.parquet"]:
    print(f, (out / f).stat().st_size, "bytes")

weather.csv 1350 bytes
weather.json 4276 bytes
weather.parquet 4616 bytes


In [23]:
csv_df = pd.read_csv(out / "weather.csv")
pq_df = pd.read_parquet(out / "weather.parquet")

print("CSV:    ", csv_df.dtypes["time"])
print("Parquet:", pq_df.dtypes["time"])

CSV:     str
Parquet: datetime64[us]
